# Intervention Study — Plot Results

Reads `intervention_raw.csv` and produces:
1. **`verdict_rates.png`** — P(A>B) by echo condition (overall)
2. **`verdict_rates_stratified.png`** — P(A>B) by echo condition, stratified by `one_correct`

In [ ]:
import sys
from pathlib import Path

# Ensure pp_experiments/ is on the path when running from the notebook's directory
HERE = Path(".").resolve()
PP_EXP = HERE.parent if HERE.name == "intervention_study" else HERE
if str(PP_EXP) not in sys.path:
    sys.path.insert(0, str(PP_EXP))

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — edit these to switch dataset / judge / intervention
# ---------------------------------------------------------------------------
from config import DEFAULT_DATASET, DEFAULT_JUDGE, DEFAULT_INTERVENTION

DATASET      = DEFAULT_DATASET       # e.g. "bbh"
JUDGE        = DEFAULT_JUDGE         # e.g. "openai/o3"
INTERVENTION = DEFAULT_INTERVENTION  # "add_echo" or "remove_echo"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from utils import get_results_dir
from causal_analysis import wald_rate

mpl.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "legend.title_fontsize": 14,
    "lines.linewidth": 1.5,
})

ECHO_ORDER   = [-1, 0, 1]
ECHO_PALETTE = {-1: "C0", 0: "C1", 1: "C2"}
ECHO_LABELS  = {
    -1: "echo=−1\n(add_echo to B)",
     0: "echo= 0\n(original)",
     1: "echo=+1\n(add_echo to A)",
}

results_dir = get_results_dir(PP_EXP / "intervention_study" / "results", DATASET, JUDGE) / INTERVENTION
raw_path    = results_dir / "intervention_raw.csv"

print(f"Loading {raw_path} ...")
df = pd.read_csv(raw_path, dtype={"question_id": str})
df = df.dropna(subset=["preference"]).copy()
df["preference"] = df["preference"].astype(int)
print(f"Loaded {len(df)} usable rows")

df.head()

In [ ]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def compute_rates(data, echo_order=ECHO_ORDER):
    """Return DataFrame with columns: echo_condition, verdict_rate, ci_lo, ci_hi, n."""
    records = []
    for ec in echo_order:
        vals = data[data["echo_condition"] == ec]["preference"].values.astype(float)
        if len(vals) == 0:
            continue
        rate, lo, hi = wald_rate(vals)
        records.append({"echo_condition": ec, "verdict_rate": rate, "ci_lo": rate - lo, "ci_hi": hi - rate, "n": len(vals)})
    return pd.DataFrame(records)


def _save(fig, name):
    path = results_dir / name
    fig.savefig(path, bbox_inches="tight", dpi=150)
    print(f"Saved → {path}")

## 1. Overall P(A>B) by echo condition

In [ ]:
rates = compute_rates(df)

fig, ax = plt.subplots(figsize=(8, 5))
colors = [ECHO_PALETTE[ec] for ec in rates["echo_condition"]]
x_labels = [ECHO_LABELS[ec] for ec in rates["echo_condition"]]

ax.bar(
    x_labels,
    rates["verdict_rate"],
    color=colors,
    alpha=0.8,
    yerr=[rates["ci_lo"], rates["ci_hi"]],
    capsize=5,
    error_kw={"elinewidth": 1.3},
)
ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8, alpha=0.7)
ax.set_ylim(0, 1.05)
ax.set_ylabel("P(A>B)")
# ax.set_title(f"Intervention study ({INTERVENTION}) — P(A>B) by echo condition\n"
#              f"Judge: {JUDGE}  |  Dataset: {DATASET}  |  n={len(df)}")

ax.set_title(f"P(A>B) depending on echo (n=500 per condition)")

# Annotate value per bar
for i, row in rates.iterrows():
    v = row["verdict_rate"]
    ax.text(i, row["verdict_rate"] + row["ci_hi"] + 0.03, f"{v:.2f}",
            ha="center", va="bottom", fontsize=9)

fig.tight_layout()
_save(fig, "verdict_rates.png")
plt.show()

## 2. P(A>B) stratified by `one_correct`

In [ ]:
og_df = df[df["intervention"] == "original"]
n_a = len(og_df[og_df["one_correct"] == 0])
n_b = len(og_df[og_df["one_correct"] == 1])
oc_labels = {
    0: f"both/neither correct\n(n={n_a})",
    1: f"one correct\n(n={n_b})",
}
oc_vals = sorted(df["one_correct"].unique())

records = []
for oc in oc_vals:
    sub = df[df["one_correct"] == oc]
    for ec in ECHO_ORDER:
        vals = sub[sub["echo_condition"] == ec]["preference"].values.astype(float)
        if len(vals) == 0:
            continue
        rate, lo, hi = wald_rate(vals)
        records.append({
            "echo_condition": ec,
            "one_correct": oc,
            "verdict_rate": rate,
            "ci_lo": rate - lo,
            "ci_hi": hi - rate,
            "n": len(vals),
        })

strat_df = pd.DataFrame(records)

# Grouped bar chart — group by one_correct, bars colored by echo condition
n_ec  = len(ECHO_ORDER)
width = 0.8 / n_ec
x     = np.arange(len(oc_vals))

fig, ax = plt.subplots(figsize=(8, 5))

for j, ec in enumerate(ECHO_ORDER):
    sub    = strat_df[strat_df["echo_condition"] == ec].set_index("one_correct").reindex(oc_vals)
    offset = (j - n_ec / 2 + 0.5) * width
    yerr   = np.array([sub["ci_lo"].fillna(0), sub["ci_hi"].fillna(0)])
    ax.bar(
        x + offset,
        sub["verdict_rate"],
        width * 0.9,
        label=ECHO_LABELS[ec].replace("\n", " "),
        color=ECHO_PALETTE[ec],
        alpha=0.8,
        yerr=yerr,
        capsize=4,
        error_kw={"elinewidth": 1.2},
    )
    # Annotate value per bar
    for i, oc in enumerate(oc_vals):
        v = sub.loc[oc, "verdict_rate"]
        if np.isnan(v):
            continue
        err = sub.loc[oc, "ci_hi"] if not np.isnan(sub.loc[oc, "ci_hi"]) else 0
        ax.text(x[i] + offset, v + err + 0.03, f"{v:.2f}",
                ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([oc_labels[oc] for oc in oc_vals])
ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8, alpha=0.7)
ax.set_ylim(0, 1.05)
ax.set_ylabel("P(A>B)")
# ax.set_title(f"Intervention study ({INTERVENTION}) — P(A>B) by echo condition, stratified by one_correct\n"
#              f"Judge: {JUDGE}  |  Dataset: {DATASET}")
# ax.legend(title="echo condition", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_title("P(A>B|echo) depending on type of pair")

fig.tight_layout()
_save(fig, "verdict_rates_stratified.png")
plt.show()